# Explore HELPMed Chats

Interactive viewer for `ambean/HELPMed` chat transcripts.

In [1]:
import ast
import ipywidgets as widgets
from datasets import load_dataset
from IPython.display import Markdown, display, clear_output

examples_dataset = load_dataset(
    "csv",
    data_files={"examples": "hf://datasets/ambean/HELPMed/examples.csv"},
    split="examples",
)
dataframe = examples_dataset.to_pandas()
dataframe = dataframe[dataframe["chat_history"].notna()].copy()
dataframe = dataframe[dataframe["chat_history"].astype(str).str.startswith("{")].copy()
dataframe = dataframe.reset_index(drop=True)

display(dataframe.head(3))
print(f"Rows: {len(dataframe)}")
print(f"Columns: {list(dataframe.columns)}")

,generated_datetime,scenario_id,id,treatment_id,controlText,historical_responses_model,chat_history,likely_cause,next_step,next_step_conf,participant_id
0,2024-09-04 14:52:19,489599,464400,1,None,"[{'iteration': 0, 'responses_model': [{'id': 0...","{'user': [{'id': 1, 'text': 'Severe headache, ...","Slurred speech and severe headache, neck pain ...",A&E (I need emergency hospital treatment),73,0
1,2024-09-04 14:55:52,489599,464415,1,None,"[{'iteration': 0, 'responses_model': [{'id': 0...","{'user': [{'id': 1, 'text': 'I have severe hea...",I believe that initially it is a migraine and ...,"Urgent Primary Care (I should be seen today, b...",64,1
2,2024-09-04 15:00:33,489599,464481,1,None,"[{'iteration': 0, 'responses_model': [{'id': 0...","{'user': [{'id': 1, 'text': ""i'M A MAN AGED 20...",Suspected brain issue,"Urgent Primary Care (I should be seen today, b...",74,2


Rows: 1800
Columns: ['generated_datetime', 'scenario_id', 'id', 'treatment_id', 'controlText', 'historical_responses_model', 'chat_history', 'likely_cause', 'next_step', 'next_step_conf', 'participant_id']


In [2]:
dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   generated_datetime          1800 non-null   object
 1   scenario_id                 1800 non-null   int64 
 2   id                          1800 non-null   int64 
 3   treatment_id                1800 non-null   int64 
 4   controlText                 0 non-null      object
 5   historical_responses_model  1800 non-null   object
 6   chat_history                1800 non-null   object
 7   likely_cause                1800 non-null   object
 8   next_step                   1800 non-null   object
 9   next_step_conf              1800 non-null   int64 
 10  participant_id              1800 non-null   int64 
dtypes: int64(5), object(6)
memory usage: 154.8+ KB


In [3]:
def parse_chat_history(chat_history_text):
    """Parse HELPMed chat_history string into a dictionary."""
    parsed_value = ast.literal_eval(chat_history_text)
    if not isinstance(parsed_value, dict):
        raise TypeError("chat_history is not a dictionary after parsing")
    return parsed_value


def format_turns(parsed_chat):
    """Create readable User and LLM turns from parsed chat data."""
    user_turns = {}
    for item in parsed_chat["user"]:
        user_turns[item["id"]] = item["text"].strip()

    llm_turns = {}
    for item in parsed_chat["bot"]:
        llm_turns[item["id"]] = item["text"].strip()

    turn_ids = sorted(set(user_turns) | set(llm_turns))
    lines = []
    for turn_id in turn_ids:
        if turn_id in user_turns:
            lines.append(f"**User ({turn_id})**\n\n{user_turns[turn_id]}")
        if turn_id in llm_turns:
            lines.append(f"**LLM ({turn_id})**\n\n{llm_turns[turn_id]}")

    return "\n\n---\n\n".join(lines)


dataframe["chat_history_parsed"] = dataframe["chat_history"].map(parse_chat_history)
dataframe["chat_readable"] = dataframe["chat_history_parsed"].map(format_turns)

In [4]:
def flatten_text(parsed_chat):
    """Join all User and LLM texts for text search."""
    user_texts = [item["text"] for item in parsed_chat["user"]]
    llm_texts = [item["text"] for item in parsed_chat["bot"]]
    return " ".join(user_texts + llm_texts)


dataframe["search_text"] = dataframe["chat_history_parsed"].map(flatten_text)

index_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(dataframe) - 1,
    step=1,
    description="Row",
    continuous_update=False,
    readout=True,
)
search_box = widgets.Text(description="Search", placeholder="keyword")
matches = widgets.Select(options=[], rows=8, description="Matches")
output = widgets.Output()


def build_match_options(query_text):
    """Return (label, index) options for rows containing the query."""
    query = query_text.lower().strip()
    if not query:
        return []

    options = []
    for row_index, row in dataframe.iterrows():
        haystack = row["search_text"].lower()
        if query in haystack:
            short_preview = row["search_text"].replace("\n", " ").strip()[:90]
            label = f"{row_index}: {short_preview}"
            options.append((label, row_index))

    return options


def render_row(row_index):
    """Render one conversation with User and LLM turns."""
    row = dataframe.iloc[row_index]
    with output:
        clear_output(wait=True)
        display(Markdown(f"### Example {row_index}"))
        display(Markdown(row["chat_readable"]))


def on_slider_change(change):
    """Update conversation display when selected row changes."""
    if change["name"] == "value":
        render_row(change["new"])


def on_search_change(change):
    """Update match list based on search text."""
    if change["name"] == "value":
        matches.options = build_match_options(change["new"])


def on_match_change(change):
    """Jump to selected match in the row slider."""
    if change["name"] == "value" and change["new"] is not None:
        index_slider.value = change["new"]


index_slider.observe(on_slider_change, names="value")
search_box.observe(on_search_change, names="value")
matches.observe(on_match_change, names="value")

controls = widgets.VBox([search_box, matches, index_slider])
display(controls, output)
render_row(index_slider.value)

Output()

## Look into ground thruth

In [19]:
load_dataset(
    "csv",
    data_files={"examples": "hf://datasets/ambean/HELPMed/background.csv"},
    # split="examples",
)

background.csv: 0.00B [00:00, ?B/s]

Generating examples split: 0 examples [00:00, ? examples/s]

DatasetDict({
    examples: Dataset({
        features: ['StartDate', 'EndDate', 'Duration (in seconds)', 'RecordedDate', '18+', 'Consent', 'Education', 'English - Speak', 'English - Read', 'English - Write', 'Online - News', 'Online - Travel', 'Online - Health', 'NHS', 'Medical Practice', 'LLMs - Drafting', 'LLMs - Info', 'LLMs - Health Info', 'LLMs - Code', 'LLMs - Editing', 'LLMs - Images', 'Which LLMs', 'Which LLMs - OTHER', 'TREATMENT_ID', 'participant_id', 'response_id'],
        num_rows: 1298
    })
})